# Apprentice on Google Colab

在 Colab 上跑 `apprentice/` 的 PPO 訓練。免費 T4 即可，不需要 24 小時不間斷。

**持久化策略**：把 `apprentice/models` 與 `apprentice/runs` 軟連結到 Google Drive，斷線重連時 `--load-model auto` 就能續跑（VecNormalize + curriculum sidecar 一起帶回來）。

**執行順序**：由上往下逐格執行即可。第一次跑時請在第 2 格授權掛載 Drive。

## 1. 檢查 GPU

In [ ]:
!nvidia-smi

## 2. 掛載 Google Drive（持久化 models/runs）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/sudoku_apprentice'
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/runs',   exist_ok=True)
print('Drive root:', DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

## 3. Clone 專案

Repo 不大（含 `data/puzzle_pool.db` 約 50 MB），直接 clone 到 `/content/sudoku_old/`。
若已存在則執行 `git pull`。

In [ ]:
REPO_URL = 'https://github.com/leo53021313/sudoku_old.git'
REPO_DIR = '/content/sudoku_old'

import os, subprocess
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('[info] repo exists, pulling…')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

%cd {REPO_DIR}
!git log --oneline -3
!ls -lh data/puzzle_pool.db

## 4. 安裝依賴

In [ ]:
!pip install -q -r apprentice/requirements.txt
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 5. 把 `apprentice/models` 與 `apprentice/runs` 連到 Drive

這樣 checkpoint 與 TensorBoard log 都會直接寫進 Google Drive，Colab session 結束也不會掉。

In [ ]:
import os, shutil

def link_to_drive(local_rel: str, drive_sub: str):
    local_abs = os.path.join(REPO_DIR, local_rel)
    drive_abs = os.path.join(DRIVE_ROOT, drive_sub)
    os.makedirs(drive_abs, exist_ok=True)
    if os.path.islink(local_abs):
        os.unlink(local_abs)
    elif os.path.isdir(local_abs):
        # 若是 clone 後產生的空資料夾就刪掉換成 symlink
        if not os.listdir(local_abs):
            os.rmdir(local_abs)
        else:
            # 把已有內容搬到 Drive，避免覆蓋
            for fn in os.listdir(local_abs):
                src = os.path.join(local_abs, fn)
                dst = os.path.join(drive_abs, fn)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(local_abs)
    os.symlink(drive_abs, local_abs)
    print(f'{local_abs} -> {drive_abs}')

link_to_drive('apprentice/models', 'models')
link_to_drive('apprentice/runs',   'runs')
!ls -la apprentice/models apprentice/runs

## 6. 啟動 TensorBoard

在開始訓練前先掛上 TB，之後 cell 7 的訓練輸出會即時刷新。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir apprentice/runs

## 7. 啟動訓練

參數說明：
- `--n-envs 4`：Colab 免費版只有 ~2 vCPU，4 個 SubprocVecEnv 已經夠用。
- `--device cuda`：直接吃 T4。
- `--load-model auto`：自動找 Drive 裡的 newest `apprentice_ckpt_*_steps.zip`；沒有就 cold start。
- `--timesteps 2000000`：目標總步數；不是「再跑這麼多」，是「停在這個數字」。要繼續就把這個調大。

如果 Colab 在訓練中途斷線，重連後重新由 cell 2 開始執行即可，會從最新 checkpoint 接續。

In [ ]:
!python -m apprentice.train.train \
    --timesteps 2000000 \
    --n-envs 4 \
    --device cuda \
    --load-model auto

## 8. （選用）驗證/查看最新 checkpoint

In [ ]:
!ls -lh apprentice/models/ | tail -10